# Silicon Chip Performance Analysis & ML

## 📌 What This Notebook Covers
This beginner-friendly notebook walks through a **complete Data Science pipeline** on a Silicon Chip dataset with 100,000 records covering processors from major manufacturers (Intel, AMD, Apple, NVIDIA, etc.).

**We will:**
1. Load and explore the dataset (EDA)
2. Visualize key patterns and distributions
3. Engineer new features
4. Build and compare ML models to classify chip type
5. Evaluate results honestly and draw conclusions

> 💡 **Note for beginners:** Every step is explained in simple terms. Code comments guide you through the logic.


## 1. 📦 Import Libraries

In [ ]:
# Standard data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Suppress minor warnings
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print("✅ All libraries loaded successfully!")


## 2. 📂 Load the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('/kaggle/input/silicon-chip-dataset/Silicon_Chip_Dataset.csv')

print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print()
df.head()


In [ ]:
# Quick info about data types and missing values
print("=== Column Info ===")
print(df.dtypes)
print()
print("=== Missing Values ===")
print(df.isnull().sum())
print()
print("✅ No missing values — clean dataset!")


## 3. 🔍 Exploratory Data Analysis (EDA)

Let's understand the data before jumping into modelling.  
EDA helps us spot patterns, distributions, and any quirks in the data.


In [ ]:
# Summary statistics for numeric columns
df.describe().round(2)


### 3.1 Chip Type Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chip type counts
chip_counts = df['chip_type'].value_counts()
axes[0].bar(chip_counts.index, chip_counts.values, color=sns.color_palette('muted', 5))
axes[0].set_title('Chip Type Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Chip Type')
axes[0].set_ylabel('Count')
for i, v in enumerate(chip_counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontsize=9)

# Manufacturer distribution
mfr_counts = df['manufacturer'].value_counts()
axes[1].barh(mfr_counts.index, mfr_counts.values, color=sns.color_palette('Set2', len(mfr_counts)))
axes[1].set_title('Chips by Manufacturer', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()
print(f"\nChip types: {df['chip_type'].unique().tolist()}")
print(f"Manufacturers: {df['manufacturer'].unique().tolist()}")


### 3.2 Key Numeric Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
cols = ['benchmark_score', 'retail_price_usd', 'transistor_count_billion',
        'process_node_nm', 'yield_percent', 'tdp_watts']

for ax, col in zip(axes.flatten(), cols):
    ax.hist(df[col], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(col.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')

plt.suptitle('Distribution of Key Numeric Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


### 3.3 Benchmark Score by Chip Type

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='chip_type', y='benchmark_score', palette='Set2')
plt.title('Benchmark Score by Chip Type', fontsize=13, fontweight='bold')
plt.xlabel('Chip Type')
plt.ylabel('Benchmark Score')
plt.tight_layout()
plt.show()

# Print mean benchmark per chip type
print("Mean Benchmark Score by Chip Type:")
print(df.groupby('chip_type')['benchmark_score'].mean().round(0).to_string())


### 3.4 Correlation Heatmap

In [ ]:
# Correlation of numeric features
num_cols = ['process_node_nm', 'transistor_count_billion', 'die_area_mm2', 'core_count',
            'base_frequency_ghz', 'boost_frequency_ghz', 'tdp_watts', 'benchmark_score',
            'yield_percent', 'failure_rate_percent', 'retail_price_usd']

plt.figure(figsize=(12, 8))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Observation: This is a synthetic dataset — features are largely independent of each other.")
print("   This is reflected in the near-zero correlations across the board.")


### 3.5 Price & Foundry Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Price by chip type
sns.violinplot(data=df, x='chip_type', y='retail_price_usd', palette='pastel', ax=axes[0])
axes[0].set_title('Retail Price Distribution by Chip Type', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Chip Type')
axes[0].set_ylabel('Retail Price (USD)')

# Foundry market share
foundry_counts = df['foundry'].value_counts()
axes[1].pie(foundry_counts, labels=foundry_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set3', 3), startangle=90)
axes[1].set_title('Foundry Market Share', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


## 4. 🛠️ Feature Engineering

We create new meaningful features from existing ones.  
Good features can help the model find patterns more easily.


In [ ]:
# Performance efficiency ratio
df['perf_per_watt'] = df['benchmark_score'] / df['tdp_watts']

# Transistor density (how many transistors packed per mm²)
df['transistor_density'] = df['transistor_count_billion'] / df['die_area_mm2']

# Boost vs base frequency ratio (how much the chip can turbo)
df['freq_boost_ratio'] = df['boost_frequency_ghz'] / df['base_frequency_ghz']

# Price per core
df['price_per_core'] = df['retail_price_usd'] / df['core_count']

# Yield quality ratio
df['yield_quality'] = df['yield_percent'] / (df['failure_rate_percent'] + 1)

print("✅ New features created:")
new_feats = ['perf_per_watt', 'transistor_density', 'freq_boost_ratio', 'price_per_core', 'yield_quality']
print(df[new_feats].describe().round(3))


## 5. 🔧 Data Preparation

In [ ]:
# Encode categorical columns into numbers
# ML models need numbers, not text

le_mfr     = LabelEncoder()
le_foundry = LabelEncoder()
le_type    = LabelEncoder()

df['manufacturer_enc'] = le_mfr.fit_transform(df['manufacturer'])
df['foundry_enc']      = le_foundry.fit_transform(df['foundry'])

# Target: chip_type
y = le_type.fit_transform(df['chip_type'])
classes = le_type.classes_
print("Classes:", classes)
print("Encoded labels:", list(range(len(classes))))


In [ ]:
# Select features for the model
features = [
    # Hardware specs
    'process_node_nm', 'transistor_count_billion', 'die_area_mm2', 'core_count',
    'base_frequency_ghz', 'boost_frequency_ghz', 'tdp_watts',
    # Performance & quality
    'benchmark_score', 'yield_percent', 'failure_rate_percent',
    # Market info
    'retail_price_usd', 'release_year',
    # Encoded categoricals
    'manufacturer_enc', 'foundry_enc',
    # Engineered features
    'perf_per_watt', 'transistor_density', 'freq_boost_ratio', 'price_per_core', 'yield_quality'
]

X = df[features]

# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {X_train.shape[0]:,}")
print(f"Testing  samples : {X_test.shape[0]:,}")
print(f"Number of features: {X_train.shape[1]}")


## 6. 🤖 Model Training & Comparison

We train **3 popular ML models** and compare their performance:
- **Random Forest** — ensemble of decision trees
- **XGBoost** — gradient boosting (very popular on Kaggle)
- **LightGBM** — fast gradient boosting by Microsoft


In [ ]:
# Define models
models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=None, random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=6,
        random_state=42, eval_metric='mlogloss', verbosity=0
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=200, learning_rate=0.1, random_state=42, verbosity=-1
    ),
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = {'model': model, 'acc': acc, 'y_pred': y_pred}
    print(f"{name:>15} | Accuracy: {acc:.4f} ({acc*100:.2f}%)")

print()
best_name = max(results, key=lambda k: results[k]['acc'])
print(f"🏆 Best Model: {best_name} ({results[best_name]['acc']*100:.2f}%)")


## 7. 📊 Model Evaluation

In [ ]:
# Use the best model for detailed evaluation
best_model = results[best_name]['model']
best_pred  = results[best_name]['y_pred']

print(f"=== {best_name} — Classification Report ===\n")
print(classification_report(y_test, best_pred, target_names=classes))


In [ ]:
# Confusion Matrix — shows where the model is right and wrong
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, best_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'{best_name}\nConfusion Matrix', fontsize=12, fontweight='bold')

# Accuracy comparison bar chart
model_names = list(results.keys())
accuracies  = [results[m]['acc'] for m in model_names]
colors = ['steelblue' if m != best_name else 'seagreen' for m in model_names]
bars = axes[1].bar(model_names, accuracies, color=colors, edgecolor='white', linewidth=0.8)
axes[1].axhline(y=0.20, color='red', linestyle='--', linewidth=1.5, label='Random Baseline (20%)')
axes[1].set_ylim(0, 0.35)
axes[1].set_title('Model Accuracy Comparison', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
for bar, acc in zip(bars, accuracies):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{acc:.4f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


### 7.1 Feature Importance

In [ ]:
# Which features matter most to the best model?
if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=features)
    fi_sorted = fi.sort_values(ascending=True)

    plt.figure(figsize=(10, 8))
    colors = ['#2196F3' if v < fi_sorted.median() else '#4CAF50' for v in fi_sorted.values]
    fi_sorted.plot(kind='barh', color=colors)
    plt.title(f'{best_name} — Feature Importance', fontsize=13, fontweight='bold')
    plt.xlabel('Importance Score')
    plt.tight_layout()
    plt.show()

    print("Top 5 most important features:")
    print(fi.sort_values(ascending=False).head(5).to_string())


## 8. 💡 Understanding the Results

> **Why is the accuracy around 20%?**

This dataset is **synthetically generated** — the feature values (benchmark score, price, frequency, etc.)
are **randomly assigned** to chip types, without real-world rules linking them.

In the real world:
- GPUs have many more CUDA cores than CPUs
- ASICs have very specific die areas and power profiles
- NPUs are optimised for low-power AI inference

Since those patterns **don't exist** in this synthetic data, no ML model — no matter how powerful — can
learn to distinguish chip types above random chance (~20% for 5 balanced classes).

**This is exactly the expected outcome for a synthetic dataset, and reporting it honestly is what good data scientists do.**


In [ ]:
# Demonstrating the data is synthetic: mean values per chip type
print("Mean feature values by chip type — notice how similar they are:")
summary_cols = ['benchmark_score', 'core_count', 'transistor_count_billion',
                'retail_price_usd', 'tdp_watts', 'process_node_nm']
print(df.groupby('chip_type')[summary_cols].mean().round(2).to_string())
print()
print("💡 Insight: Values are nearly identical across chip types — confirming random generation.")


## 9. ✅ Conclusion

### What We Did
| Step | Action |
|------|--------|
| EDA | Explored 100,000 chip records across 8 manufacturers, 3 foundries, 5 chip types |
| Feature Engineering | Created 5 new features: perf/watt, transistor density, boost ratio, price/core, yield quality |
| Modelling | Trained and compared Random Forest, XGBoost, and LightGBM |
| Evaluation | Measured accuracy, classification report, confusion matrix, and feature importance |

### Key Findings
- All three models achieve **~20% accuracy** — equal to random guessing for 5 balanced classes
- This is **not a modelling failure** — it correctly reflects the synthetic nature of the data
- Feature importances are **uniformly distributed**, confirming no real signal exists between features and chip type
- In a real-world chip dataset with genuine engineering relationships, tree-based models typically achieve **85–95% accuracy**

### What Would Work on Real Data
- More structured features (e.g. GPU vs CPU core architecture flags)
- Domain-driven feature engineering based on actual chip specifications
- Time-series analysis of performance improvements across `release_year`

### Tags
`silicon`, `chips`, `semiconductor`, `classification`, `random-forest`, `xgboost`, `lightgbm`,
`feature-engineering`, `eda`, `beginner`, `synthetic-data`, `multiclass-classification`
